In [1]:
import sys, os
sys.path.append('./../')
import torch
import numpy as np
import time
from config import absjoin, PANDA_XML, SHELF_URDF, BASE_URDF


from envs.env_panda_shelf_gs import PandaShelf
 
from panda_kinematics import PandaKinematics

from src.ttts_augment import TTTS
from robot_utils import Point2PointMotion


%load_ext autoreload
%autoreload 2

device = 'cuda' if torch.cuda.is_available() else 'cpu'

[I 07/03/25 11:29:52.343 189093] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [2]:
ARM_BASE_HEIGHT = 0.7

MAX_BATCH = 100

In [3]:
def env_infos_gen():
    """
    This function generates the environment information 
    useful for defining the simulation environment.
    """
    shelf_info = {
        "name": "shelf",
        "urdf": SHELF_URDF,
        "base_pose": [0.7, 0.0, 0.015, 0, 0, 90],
    }
    print("SHELF_URDF: ", SHELF_URDF)

    base_info = {
        "name": "base",
        "urdf": BASE_URDF,
        "base_pose": [0.0, 0.0, 0.35, 0, 0, 90]
    }

    robot_info = {
        "name": "panda",
        "xml": PANDA_XML,
        "base_pose": [0.0, 0.0, 0.8, 0, 0, 0],
        # "conf": [0.0, -np.pi/4, 0.0, -np.pi/2, 
        #          0, np.pi/2, np.pi/4, 0.06, 0.06],
        "conf": [1.76, -4.5e-01, -1.77, -1.35,  4.9e-01,
                2.36, -2.8e-01, 0.06, 0.06],
        "jnt_names": ["joint1", "joint2", "joint3",
                    "joint4", "joint5", "joint6",
                    "joint7", "finger_joint1", "finger_joint2", ],
        "kp": np.array([4500, 4500, 3500, 3500, 2000, 2000, 
                            2000, 100, 100]),
        "kv": np.array([450, 450, 350, 350, 200, 200, 200, 
                        10, 10]),
        "force_range": {
            "lower": np.array([-87, -87, -87, -87, -12, -12, 
                                -12, -100, -100]),
            "upper": np.array([87,  87,  87, 87,  12,  12, 
                                12,  100,  100]),
        },
    }

    

    utilities = {
        "target_x1": [0.70, -0.15, 1.5],
        "target_x2": [0.60, 0.15, 1.1],
        # "1": [0.60, 0.15, 1.5],
        # "2": [0.55, -0.2, 1.6],
        # "3": [0.5, -0.2, 1.2],
        # "4": [0.6, 0.15, 1.1],
        # "5": [0.7, 0.1, 1.1],
        # "6": [0.6, -0.1, 1.1],
    }

    env_infos = {
        "shelf": shelf_info,
        "base": base_info,
        "robot": robot_info,
        "utilities": utilities
    }

    return env_infos

def test_collision(env_infos):
    action1 = [[0.0, -np.pi/4, 0.0, -np.pi/2, 
                 0, np.pi/2, np.pi/4, 0.06, 0.06]]*2
    action2 = [[0.0, -np.pi/4, 0.0, -np.pi/2, 
                 0, np.pi/2, np.pi/4, 0.06, 0.06]]*8
    
    action = action1 + action2
    action = torch.tensor(action, dtype=torch.float32)

    return action



In [4]:
# Define CMA-ES parameters
num_guess = 5
popsize = 100
num_iterations = 10
# n_envs = num_guess * popsize
max_batch = MAX_BATCH
K = 2

# Define p2p motion
T = 1
dt = 0.01
n_joints = 7
basis = 'bs'
bounds = None
p2p_motion = Point2PointMotion(T=T, dt=dt, K=K, basis=basis, n=n_joints,
                            bounds=bounds, device=device)


In [5]:
# Generate the environment information
env_infos = env_infos_gen()

In [7]:
sol_2 = torch.load("IK_sol_2.pth", map_location=device)
env = PandaShelf(n_envs=sol_2.shape[0], gpu=True, use_gui=True,
                p2p_motion=p2p_motion)
env.load_world(env_infos, render_utility=True)

env.reset(env_infos)


robot = env_infos["robot"]["id"]
eef = robot.get_link("grasping_target_hand")
eef_pos = eef.get_pos()

env.set_robot_configuration(robot, sol_2[:, :].cpu().numpy(), env_infos["robot"])
# env.step(robot, sol_2[:, :].cpu().numpy())



[Genesis] [11:30:00] [WARNING] Overriding base link's pos with user provided value in morph.
[Genesis] [11:30:00] [WARNING] Overriding base link's quat with user provided value in morph.


In [8]:
eef = robot.get_link("grasping_target_hand")
eef.get_pos()

tensor([[0.6000, 0.1500, 1.1000],
        [0.6000, 0.1500, 1.1000],
        [0.6000, 0.1501, 1.1000],
        [0.6000, 0.1500, 1.0999],
        [0.5999, 0.1500, 1.1001]], device='cuda:0', dtype=torch.float32)